In [ ]:
#Importing the relevant libraries and tools from those libraries needed to get our tool and its functions to run.

import transformers
import torch
import langchain_community
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoProcessor, AutoModelForMultimodalLM
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
import pypdf
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_chroma import Chroma
import gradio as gr
from pylatexenc.latex2text import LatexNodes2Text

/Users/danclancy/Desktop/Dissertation Folder/RAG LLM Code Area/Dissertation_Repository/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/0f/9853n3y529v8dtlr3sfn_zf00000gn/T/ipykernel_42972/2701672036.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  import langchain_community


In [ ]:
#Loading in of the gemma E4B model from huggingface. This was loaded in using the recommendations given on the Gemma E4B huggingface page. 
# I also set up the processor and tokenizer for text decoding and processing using the gemma model to ensure all components ran well in conjuction with one another

model_name = "google/gemma-4-E4B-it"

processor = AutoProcessor.from_pretrained(model_name)

#Dtype auto sets the preferred data type automatically to whatever is preferred by the system.
#Device map was set to mps in order to utilize my laptops M chip (this is specifically for Macbook.).
model = AutoModelForMultimodalLM.from_pretrained(
    model_name,
    dtype="auto",
    device_map="mps",
    
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

Loading weights: 100%|██████████| 2076/2076 [00:00<00:00, 2326.23it/s]


In [ ]:
#embedding section
#Setup our huggingface embeddings model again with recommendations from langchain.
#Used the embedding gemma 300m parameters embedding model which was to be utilised by gemma-4 models for embedding text.
#Normalised embeddings to ensure equal weightings of all embeddings to prevent unneccessary bias coming into play.
embeddingsTWO = HuggingFaceEmbeddings(
    model_name="google/embeddinggemma-300m",
    encode_kwargs={"normalize_embeddings": True},
)

Loading weights: 100%|██████████| 314/314 [00:00<00:00, 8104.24it/s]


In [ ]:
#Using Chroma to setup our persistant vector store to store our document embeddings in between running the repository.
#Using our previously setup embeddings model as the embedding model for our database.
#Setting up our persist directory so thats where our embedding gets stored and pulled from every-time we run the script.
vector_store = Chroma(
    collection_name="LocalEmbeddingsGemma300m",
    embedding_function=embeddingsTWO,
    persist_directory="./chroma_RAG_db"
)

In [ ]:
#Setup a text splitter using RecursiveCharacterTextSplitter from langchain.
#Setting chunksize to 1000 to ensure I am capturing a large enough chunk of text for context.
#Setting the chunk overlap to 200 to ensure enough overlap between chunks so context is not lost.
#Adding start index per Langcahin to add metadata denoting the start of the text chunk in regards to base document.
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000, chunk_overlap=200, add_start_index= True
)

In [ ]:
#Process that was used to load in documents from our document folder into the embedding store.
#Using PyPDF loader from langchain to load in the pdfs
#Using our pre setup splitter to split the documents
#Then finally we add the split documents to the vector store.
loader = PyPDFLoader("/Users/danclancy/Desktop/Dissertation Folder/Astrometry and geodesy with radio interferometry- experiments, models, results.pdf")
document = loader.load()
docSplit = splitter.split_documents(document)
vector_store.add_documents(docSplit)



In [ ]:
# RAG full prompt template with RAG formula in there.
# "Role: You are a physics expert giving a tour of your lab to a group of visitors. These visitors are all high school age and above. You keep this in mind when you are asked the user query.Task: You are to answer the user query in a complete manner. You must adapt the complexity of your answer to the complexity of the question that is asked. Tonality and Format: You are to answer in an informal friendly tone and answer with length that is as long as you see fit but no longer that a large paragraph. Context: {'\n'.join([f'-{chunk}'for chunk in results])}Constraints: You cannot by any means make up any new information. You will always tell the respondents that you do not know something if that is the case. You are only to use information given in the Context to answer the user query and no other information. YOU CAN ONLY USE INFORMATION FROM THE CONTEXT TO ANSWER THE USER QUESTION. Do not give the user any redundant information.

In [ ]:
# Feynman RAG full template 
# "Role:You are  a Physics University Tutor specializing in black holes. You are helping a group of first year university students answer a set of questions that they have been given by their lecturer. You are tasked with answering the user query. Task: There are 20 questions in total that they will need your help answering and you must actively assist them with all 20 questions. When asked a query you are to give them short and concise answers that still answers the query completely. Tonality and Format: You are to respond to the students in a friendly but professional way using clear words that are non-academic in nature and simple to understand. Context: {'\n'.join([f'-{chunk}'for chunk in results])} Constraints: Answer the questions clearly and effectively as stated above. You are only to use information given in the Context to answer the question and no other information. YOU CAN ONLY USE INFORMATION FROM THE CONTEXT TO ANSWER THE QUESTION Do not give the students any information that is redundant and that they do not need. You will always tell the respondents that you do not know something if that is the case.  Before answering the question you must do the following 4 step process ONLY ONCE. 1.You must explain your response in simple terms as if explaining it to a child. 2. You must identify gaps in your explanation or any inconsistencies in your explanation. 3. You must then re-write and refine your explanation based on these gaps or inconsistencies. 4. You must then produce the answer in a concise manner.

In [3]:
#Setting up the previous query string here as empty this is going to be utilised
previous_query=[]

In [4]:
#Key words lists setup here for the process below in order to check whether question has key words in it.
keywords = ["black hole", "blackhole","black", "hole", "event horizon","event","horizon", "singularity", "space time","spacetime","space","time", "general relativity","special relativity", "schwarzschild radius","radius","shwarzschild", "white hole", "white", "quasar", "galaxy","galaxies","supermassive","stellar mass","stellar", "kerr","star", "red-shift", "gravity","gravitational","spaghetti","spaghettification", "light","relative","X-Ray","XRay","microlensing","gravitational microlensing","accretion disk","gravitational time dilation","time dilation","Hawking Radiation","Celestial Reference Frame","VLBI","Very Long Baseline Interferometry", "Reference Frame","Celestial"]

In [ ]:
#Setting up the QA function with parameters recommended by gradio that will allow us to use the gradio interaface in the next kernel.
def QA (message,history):

#making the message lowercase so that there is no case sensitivity issues when looking for words in the keywords list
    message = message.lower()
#setting up an if statement to check to see if any of the words in the user query are in the key words list.
# Using the If we are also checking to see if the message is longer than 10 characters.
    if any(x in message for x in keywords) & (len(message)>10):
        #if both of the above are met the question is added to the previous_query list and the message is set as the user query.
        previous_query.append(message)
        message = message
    else:
        #If this is not the case then we append the previous question asked to the message.
        #For instance if asked "Explain further?" then it will be "Explain Furhter?" + the previous question -1 which is the last question asked.
        message = message + " " + previous_query[-1]

   


    results = vector_store.similarity_search(message)
    #we then do a similarity search from langchain to get the most relevant documents.
    #Using the {'\n'.join([f'-{chunk}'for chunk in results])} from class to adequately add the relevant context to our prompt template. 
    #We use the .join to add the chunks from our results which is the relevant documents from the embedding store returned by similarity search.
    
    messages = history + [{"role": "system", "content": f"""Role: You are a physics expert giving a tour of your lab to a group of visitors. These visitors are all high school age and above. You keep this in mind when you are asked the user query.
  
    Task: You are to answer the user query in a complete manner. You must adapt the complexity of your answer to the complexity of the question that is asked.
  
    Tonality and Format: You are to answer in an informal friendly tone and answer with length that is as long as you see fit but no longer that a large paragraph.
  
    Context: {'\n'.join([f'-{chunk}'for chunk in results])}
  
    Constraints: You cannot by any means make up any new information. You will always tell the respondents that you do not know something if that is the case. You are only to use information given in the Context to answer the user query and no other information. YOU CAN ONLY USE INFORMATION FROM THE CONTEXT TO ANSWER THE USER QUESTION. Do not give the user any redundant information. You must always greet the respondents with Hello.""" },{"role": "user", "content": message}]




#using the message recommendation from Gemmas Huggingface site to generate responses.
#set enable_thinking to false which was to allow us to generate responses faster with less strain on the GPU.
    input_ids = processor.apply_chat_template(messages, tokenize = False, add_generation_prompt=True, enable_thinking=False)

    inputs= processor(text=input_ids, return_tensors="pt").to(model.device)
#Setting up the input length which we use the shape function to do.
    input_len = inputs["input_ids"].shape[-1]
#Using the .generate to generate the output using the inputs setting the max_new_tokens to 1024 which allows for adequately long response times 
#while balancing it with load times.
    outputs = model.generate(**inputs, max_new_tokens=1024)
#Using the proseccor to decode the ouputs to convert the numbers generated by the model to text.
#using the outputs[0][input_len:] to only use the text from the most recent output and only decode the text that comes after the length of the input.
#This allows for cleaner outputs and was adapted from the Google Gemma Page on Huggingface.
    response = processor.decode(outputs[0][input_len:], skip_special_tokens=True)
#Then using the LatexNodes2Test library to convert all the Latex code within decode response to text.
    responselatex = LatexNodes2Text().latex_to_text(response)

    return responselatex
    
    

In [ ]:
#using the gradio chat interface alongside the QA function we setup above.
demo = gr.ChatInterface(QA, api_name="chat")
#using the .launch with the citrus theme to actually launch the chat interface through gradio that participants interacted with.
demo.launch(theme=gr.themes.Citrus())

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
